# README
This jupyter notebook was used to evaluate and check the duration of activities (cel 1) and time between activities (cel 2).

Make sure to fill in the right details (input folder, column names and threshold) to use this jupyter notebook. 
Note that the code and functionality in this notebook is not used for the S2CF+C approach, this was only for evaluation of the identification of the activities and to check whether certain assumptions about the identification were valid.

In [ ]:

VERSION = 14

import os
import pandas as pd

def time_between_events(
  df: pd.DataFrame,
  event_a: str = "7_Shutdown",
  event_b: str = "1_Startup",
  start_time_col: str = "Datetime",
  end_time_col: str = "End Datetime",
  event_col: str = "Activity"
):
  """
  Calculate min, max, mean and median times between an event A
  and an immediately following event B.

  Returns a dict with statistics and the raw deltas (in seconds).
  """

  # Ensure datetime
  df = df.copy()
  df[start_time_col] = pd.to_datetime(
    df[start_time_col], format="%d %b %Y %H:%M:%S,%f"
  )
  df[end_time_col] = pd.to_datetime(
    df[end_time_col], format="%d %b %Y %H:%M:%S,%f"
  )

  # Sort by time
  df = df.sort_values(start_time_col).reset_index(drop=True)

  # Shift columns to access the next event
  df["next_event"] = df[event_col].shift(-1)
  df["next_start"] = df[start_time_col].shift(-1)

  # Select only event A → event B transitions
  mask = (
    (df[event_col] == event_a) &
    (df["next_event"] == event_b)
  )

  transitions = df.loc[mask].copy()

  # Time delta in seconds
  transitions["delta_seconds"] = (
    transitions["next_start"] - transitions[end_time_col]
  ).dt.total_seconds()

  # Drop negative or zero gaps
  transitions = transitions[transitions["delta_seconds"] > 0]

  # Convert to list once
  timespans = transitions["delta_seconds"].tolist()

  if not timespans:
    return {
      "min": None,
      "max": None,
      "mean": None,
      "median": None,
      "count": 0,
      "timespans": []
    }

  return {
    "min": min(timespans),
    "max": max(timespans),
    "mean": sum(timespans) / len(timespans),
    "median": pd.Series(timespans).median(),
    "count": len(timespans),
    "timespans": timespans
  }


In [ ]:
# Get list of files in which the phase goes of the threshold


folder_path = f''

file_paths = [
  os.path.join(folder_path, f) for f in os.listdir(folder_path)
  if os.path.isfile(os.path.join(folder_path, f)) and f.endswith('.csv')
]

THRESHOLD_SECONDS = 60 # 1min
# THRESHOLD_SECONDS = 300 # 5min
THRESHOLD_SECONDS = 600 # 10min
# THRESHOLD_SECONDS = 3600 # 1u
# THRESHOLD_SECONDS = 10800 # 3u
TARGET_PHASE = "3_Automatic_registration"
UNDER_THRESHOLD = True
# files_exceeding_threshold = []

analyzed_ids = []

# Collect all file data in one combined dataframe
dfs = []

for file_path in file_paths:
  df = pd.read_csv(file_path)
  # print(f'check {file_path}')
  # Add metadata
  df["source_file"] = os.path.basename(file_path)

  # Parse datetimes
  df["Datetime"] = pd.to_datetime(df["Datetime"], format="%d %b %Y %H:%M:%S,%f")
  df["End Datetime"] = pd.to_datetime(df["End Datetime"], format="%d %b %Y %H:%M:%S,%f")

  # Calculate duration
  df["Duration"] = df["End Datetime"] - df["Datetime"]
  df["Duration_s"] = df["Duration"].dt.total_seconds()

  # Mask for phase + threshold
  m = df["Activity"].eq(TARGET_PHASE) & df["Duration_s"].gt(THRESHOLD_SECONDS)
  if UNDER_THRESHOLD:
    m = df["Activity"].eq(TARGET_PHASE) & df["Duration_s"].lt(THRESHOLD_SECONDS)

  if m.any():
    alert_rows = df.loc[m, ["Log ID", "Duration_s", "Datetime"]]
    # Build detailed lines for printing
    details = "\n".join(
        f"  Log ID: {row['Log ID']}, Duration: {row['Duration_s'] }s, Start: {row['Datetime']}"
        for _, row in alert_rows.iterrows() if row["Log ID"] not in analyzed_ids
    )
    print(
        # f"{file_path} has '{TARGET_PHASE}' longer than {THRESHOLD_SECONDS}s:\n"
        f"{details}"
    )

In [ ]:
import matplotlib.pyplot as plt

folder_path = f''

file_paths = [
  os.path.join(folder_path, f) for f in os.listdir(folder_path)
  if os.path.isfile(os.path.join(folder_path, f)) and f.endswith('.csv')
]

THRESHOLD_DAYS = 5  # <-- change this to whatever you need
all_timespans = []
event_a="7_Shutdown"
event_b="1_Startup"

for file_path in file_paths:
  df = pd.read_csv(file_path)
  result = time_between_events(
    df,
    event_a=event_a,
    event_b=event_b,
    start_time_col="Datetime",
    end_time_col="End Datetime",
    event_col="Activity",
  )
  if result["max"] and result["max"] > 300000:
    print(f"{file_path}: {len(result['timespans'])}, timespans: {result['timespans']}")

  all_timespans.extend(result["timespans"])

all_timespans = pd.Series(all_timespans)

final_stats = {
    "min": all_timespans.min(),
    "max": all_timespans.max(),
    "mean": all_timespans.mean(),
    "median": all_timespans.median(),
    "count": len(all_timespans)
}

print(final_stats)

plt.figure(figsize=(8, 5))
plt.hist(all_timespans, bins=20, edgecolor="black")
plt.xlabel(f"Time between {event_a} and {event_b} (seconds)")
plt.ylabel("Frequency")
plt.title(f"Distribution of time between {event_a} → {event_b}")
plt.grid(axis="y", alpha=0.3)
plt.show()

all_timespans_hours = [t / 3600 for t in all_timespans]
plt.figure(figsize=(8, 5))
plt.hist(all_timespans_hours, bins=20, edgecolor="black")
plt.xlabel(f"Time between {event_a} and {event_b} (hours)")
plt.ylabel("Frequency")
plt.title(f"Distribution of time between {event_a} → {event_b}")
plt.grid(axis="y", alpha=0.3)
plt.show()

all_timespans_days = [t / 24 for t in all_timespans_hours]
plt.figure(figsize=(8, 5))
plt.hist(all_timespans_days, bins=20, edgecolor="black")
plt.xlabel(f"Time between {event_a} and {event_b} (days)")
plt.ylabel("Frequency")
plt.title(f"Distribution of time between {event_a} → {event_b}")
plt.grid(axis="y", alpha=0.3)
plt.show()


Threshold_seconds = THRESHOLD_DAYS * 3600 * 24
below_threshold = all_timespans[all_timespans <= Threshold_seconds]
above_threshold = all_timespans[all_timespans > Threshold_seconds]

below_threshold_days = below_threshold / (3600 * 24)

print(f"Total samples: {len(all_timespans)}")
print(f"Below threshold ({THRESHOLD_DAYS} days): {len(below_threshold)}")
print(f"Filtered out (outliers): {len(above_threshold)}")

plt.figure(figsize=(8, 5))
plt.hist(below_threshold_days, bins=20, edgecolor="black")
plt.xlabel(f"Time between {event_a} and {event_b} (days)")
plt.ylabel("Frequency")
plt.title(
    f"Distribution of time between {event_a} → {event_b}\n"
    f"(values > {THRESHOLD_DAYS} days removed)"
)
plt.grid(axis="y", alpha=0.3)
plt.show()
